In [1]:
# Cell 1A — Call sleeve research setup + source inventory
# Purpose:
#   Find the existing put-pipeline files we can reuse for the short-call Excel replication.
#   No signal logic yet. No parameter tuning yet.

from pathlib import Path
import pandas as pd
import json
from datetime import datetime

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

NOTEBOOK_NAME = "01_call_sleeve_excel_replication_v1"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("Cell 1A — Call sleeve source inventory")
print("=" * 100)
print(f"Project root:   {PROJECT_ROOT}")
print(f"Audit dir:      {AUDIT_DIR}")
print(f"Processed dir:  {PROCESSED_DIR}")
print()

# Candidate files from existing put / Corsi infrastructure
candidate_paths = {
    "final_signal_base_panel": PROJECT_ROOT / "data" / "processed" / "vrp_final_signal" / "vrp_final_corsi_signal_base_panel_v1.parquet",
    "latest_snapshot": PROJECT_ROOT / "data" / "processed" / "vrp_final_signal" / "vrp_final_corsi_latest_snapshot_v1.parquet",
    "selected_put_trades": PROJECT_ROOT / "data" / "processed" / "vrp_final_signal" / "vrp_final_corsi_selected_trades_v1.parquet",
    "implied_variance_surface": PROJECT_ROOT / "data" / "processed" / "implied_variance" / "spx_vix_style_implied_variance_surface_v1.parquet",
    "spy_eod_prices": PROJECT_ROOT / "data" / "processed" / "market_data" / "spy_eod_prices_v1.parquet",
    "spy_rv_history": PROJECT_ROOT / "data" / "processed" / "market_data" / "spy_realized_vol_history_v1.parquet",
    "spy_corsi_har_input_panel": PROJECT_ROOT / "data" / "processed" / "market_data" / "spy_corsi_har_input_panel_v1.parquet",
}

rows = []

for name, path in candidate_paths.items():
    exists = path.exists()
    row = {
        "name": name,
        "path": str(path),
        "exists": exists,
        "rows": None,
        "cols": None,
        "date_min": None,
        "date_max": None,
        "tenors": None,
        "columns": None,
        "error": None,
    }
    
    if exists:
        try:
            df_head = pd.read_parquet(path)
            row["rows"] = len(df_head)
            row["cols"] = len(df_head.columns)
            row["columns"] = ", ".join(df_head.columns.astype(str).tolist())
            
            # Flexible date detection
            date_cols = [c for c in df_head.columns if c.lower() in {"date", "trade_date", "asof_date", "timestamp"}]
            if date_cols:
                dc = date_cols[0]
                dates = pd.to_datetime(df_head[dc], errors="coerce")
                row["date_min"] = dates.min()
                row["date_max"] = dates.max()
            
            # Flexible tenor detection
            tenor_cols = [c for c in df_head.columns if c.lower() in {"tenor", "tenor_d", "dte", "target_dte"}]
            if tenor_cols:
                tc = tenor_cols[0]
                tenors = sorted(pd.Series(df_head[tc]).dropna().unique().tolist())
                row["tenors"] = tenors
        
        except Exception as e:
            row["error"] = repr(e)
    
    rows.append(row)

inventory = pd.DataFrame(rows)

display_cols = [
    "name", "exists", "rows", "cols", "date_min", "date_max", "tenors", "path", "error"
]

print("Candidate source files:")
display(inventory[display_cols])

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
inventory_path = AUDIT_DIR / f"01A_call_sleeve_source_inventory_{timestamp}.csv"
inventory.to_csv(inventory_path, index=False)

print()
print(f"Saved inventory: {inventory_path}")

Cell 1A — Call sleeve source inventory
Project root:   C:\Users\patri\vrp_project
Audit dir:      C:\Users\patri\vrp_project\data\audit\call_sleeve_research
Processed dir:  C:\Users\patri\vrp_project\data\processed\call_sleeve_research

Candidate source files:


,name,exists,rows,cols,date_min,date_max,tenors,path,error
0,final_signal_base_panel,True,14715,36,2020-01-02,2026-07-08,"[9, 12, 15, 18, 21, 24, 27, 30, 33]",C:\Users\patri\vrp_project\data\processed\vrp_...,None
1,latest_snapshot,True,9,36,2026-07-08,2026-07-08,"[9, 12, 15, 18, 21, 24, 27, 30, 33]",C:\Users\patri\vrp_project\data\processed\vrp_...,None
2,selected_put_trades,True,368,21,2020-12-31,2026-06-02,"[12, 15, 21, 24, 27, 30, 33]",C:\Users\patri\vrp_project\data\processed\vrp_...,None
3,implied_variance_surface,True,18171,32,2018-06-25,2026-07-08,"[9, 12, 15, 18, 21, 24, 27, 30, 33]",C:\Users\patri\vrp_project\data\processed\impl...,None
4,spy_eod_prices,True,2139,12,2018-01-02,2026-07-08,None,C:\Users\patri\vrp_project\data\processed\mark...,None
5,spy_rv_history,True,2139,25,2018-01-02,2026-07-08,None,C:\Users\patri\vrp_project\data\processed\mark...,None
6,spy_corsi_har_input_panel,True,2139,29,2018-01-02,2026-07-08,None,C:\Users\patri\vrp_project\data\processed\mark...,None



Saved inventory: C:\Users\patri\vrp_project\data\audit\call_sleeve_research\01A_call_sleeve_source_inventory_20260709_204205.csv


In [2]:
# Cell 1B — Inspect final signal panel columns and isolate 30D call-sleeve signal inputs
# Purpose:
#   Inspect the existing Corsi final signal panel and identify the columns needed
#   for the 30D SPY short-call Excel replication.
#
# Scope:
#   - Read-only inspection
#   - 30D only
#   - No trade construction yet
#   - No payoff yet
#   - No RV21D filter for calls

from pathlib import Path
import pandas as pd
from datetime import datetime

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_SIGNAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vrp_final_signal"
    / "vrp_final_corsi_signal_base_panel_v1.parquet"
)

print("=" * 100)
print("Cell 1B — 30D call-sleeve signal input column inspection")
print("=" * 100)
print(f"Final signal path: {FINAL_SIGNAL_PATH}")
print()

df = pd.read_parquet(FINAL_SIGNAL_PATH)

print("Loaded final signal panel")
print(f"Rows:        {len(df):,}")
print(f"Columns:     {len(df.columns):,}")
print()

print("=" * 100)
print("All columns")
print("=" * 100)
for i, c in enumerate(df.columns, start=1):
    print(f"{i:02d}. {c}")

# Flexible date / tenor detection
date_candidates = ["date", "trade_date", "asof_date", "as_of_date", "timestamp", "signal_date"]
tenor_candidates = ["tenor", "tenor_d", "dte", "target_dte", "tenor_days"]

date_col = next((c for c in date_candidates if c in df.columns), None)
tenor_col = next((c for c in tenor_candidates if c in df.columns), None)

if date_col is None:
    raise ValueError(
        "Could not auto-detect date column. Review the printed column list and set date_col manually."
    )

if tenor_col is None:
    raise ValueError(
        "Could not auto-detect tenor column. Review the printed column list and set tenor_col manually."
    )

df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

print()
print("=" * 100)
print("Detected core columns")
print("=" * 100)
print(f"Date column:   {date_col}")
print(f"Tenor column:  {tenor_col}")
print(f"Date range:    {df[date_col].min().date()} to {df[date_col].max().date()}")
print(f"Tenors:        {sorted(df[tenor_col].dropna().unique().tolist())}")
print()

# 30D only
df30 = df[df[tenor_col] == 30].copy()
df30[date_col] = pd.to_datetime(df30[date_col], errors="coerce")

print("=" * 100)
print("30D slice")
print("=" * 100)
print(f"Rows:             {len(df30):,}")
print(f"Date range:       {df30[date_col].min().date()} to {df30[date_col].max().date()}")
print(f"Duplicate dates:  {df30[date_col].duplicated().sum():,}")
print()

# Candidate column detection
needed_concepts = {
    "date": [date_col],
    "tenor": [tenor_col],
    "spy_close": ["spy", "close", "underlying", "price"],
    "vix_style_vol": ["vix", "vol", "implied", "iv"],
    "forecast_variance_or_vol": ["forecast", "har", "corsi", "fds"],
    "vrp_log": ["vrp", "log"],
    "vrp_z_3m": ["z", "3m"],
    "vrp_z_1y": ["z", "1y"],
    "rsi14": ["rsi"],
}

rows = []
cols_lower = {c: c.lower() for c in df30.columns}

for concept, keywords in needed_concepts.items():
    candidates = []

    # Direct detected columns
    if concept == "date":
        candidates = [date_col]
    elif concept == "tenor":
        candidates = [tenor_col]
    else:
        for col, low in cols_lower.items():
            if all(k.lower() in low for k in keywords):
                candidates.append(col)

        # Looser fallback candidates
        if concept == "spy_close":
            for col, low in cols_lower.items():
                if (
                    low in {"close", "adj_close", "spy_close", "underlying_close", "spy_adj_close"}
                    or ("spy" in low and "close" in low)
                ) and col not in candidates:
                    candidates.append(col)

        if concept == "vix_style_vol":
            for col, low in cols_lower.items():
                if ("vix" in low or "implied" in low or "iv" in low or "vol" in low) and col not in candidates:
                    candidates.append(col)

        if concept == "vrp_log":
            for col, low in cols_lower.items():
                if "vrp" in low and col not in candidates:
                    candidates.append(col)

        if concept == "vrp_z_3m":
            for col, low in cols_lower.items():
                if ("z" in low and ("3m" in low or "63" in low)) and col not in candidates:
                    candidates.append(col)

        if concept == "vrp_z_1y":
            for col, low in cols_lower.items():
                if ("z" in low and ("1y" in low or "252" in low)) and col not in candidates:
                    candidates.append(col)

    rows.append({
        "concept": concept,
        "keywords_used": ", ".join(keywords),
        "candidate_count": len(candidates),
        "candidate_columns": ", ".join(candidates),
    })

column_inventory = pd.DataFrame(rows)

print("=" * 100)
print("Candidate columns by needed concept")
print("=" * 100)
display(column_inventory)

print()
print("=" * 100)
print("Latest 30D rows preview")
print("=" * 100)
display(df30.sort_values(date_col).tail(10))

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
audit_path = AUDIT_DIR / f"01B_call_sleeve_30d_signal_column_inventory_{timestamp}.csv"
column_inventory.to_csv(audit_path, index=False)

print()
print(f"Saved column inventory: {audit_path}")

Cell 1B — 30D call-sleeve signal input column inspection
Final signal path: C:\Users\patri\vrp_project\data\processed\vrp_final_signal\vrp_final_corsi_signal_base_panel_v1.parquet

Loaded final signal panel
Rows:        14,715
Columns:     36

All columns
01. trade_date
02. tenor
03. tenor_bucket
04. final_signal_version
05. final_signal_decision_id
06. model_spec
07. model_source
08. fit_status_candidate
09. forecast_repair_scope
10. implied_variance_final
11. vix_style_vol_final
12. forecast_variance_final
13. forecast_vol_final
14. model_vrp_log_final
15. z_3m_final
16. z_1y_final
17. rsi14_final
18. rv21d_vol_pct_final
19. spy_close
20. source_model_vrp_log
21. source_model_vrp_z_3m
22. source_model_vrp_z_1y
23. source_vs_final_model_vrp_log_diff
24. core_pass
25. secondary_pass
26. selected
27. selected_layer
28. selected_tenor_bucket
29. selected_tenor
30. selected_sleeve_id
31. selected_size_pct
32. selected_size_label
33. selected_sizing_quality_score
34. selected_win_rate
35. 

,concept,keywords_used,candidate_count,candidate_columns
0,date,trade_date,1,trade_date
1,tenor,tenor,1,tenor
2,spy_close,"spy, close, underlying, price",1,spy_close
3,vix_style_vol,"vix, vol, implied, iv",5,"implied_variance_final, vix_style_vol_final, f..."
4,forecast_variance_or_vol,"forecast, har, corsi, fds",0,
5,vrp_log,"vrp, log",5,"model_vrp_log_final, source_model_vrp_log, sou..."
6,vrp_z_3m,"z, 3m",2,"z_3m_final, source_model_vrp_z_3m"
7,vrp_z_1y,"z, 1y",2,"z_1y_final, source_model_vrp_z_1y"
8,rsi14,rsi,2,"final_signal_version, rsi14_final"



Latest 30D rows preview


,trade_date,tenor,tenor_bucket,final_signal_version,final_signal_decision_id,model_spec,model_source,fit_status_candidate,forecast_repair_scope,implied_variance_final,...,selected_layer,selected_tenor_bucket,selected_tenor,selected_sleeve_id,selected_size_pct,selected_size_label,selected_sizing_quality_score,selected_win_rate,selected_1pct_expected_loss_positive,selection_rank
14632,2026-06-23,30,Back,vrp_final_corsi_signal_panel_v1,recomputed_unified_fds_final_signal_001,unified_fds_no_min_return,unified_fds_no_min_return_oos_refit,candidate_fit,UNFROZEN_BACK_DIAGNOSTIC,0.038364,...,None,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN
14641,2026-06-24,30,Back,vrp_final_corsi_signal_panel_v1,recomputed_unified_fds_final_signal_001,unified_fds_no_min_return,unified_fds_no_min_return_oos_refit,candidate_fit,UNFROZEN_BACK_DIAGNOSTIC,0.038008,...,None,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN
14650,2026-06-25,30,Back,vrp_final_corsi_signal_panel_v1,recomputed_unified_fds_final_signal_001,unified_fds_no_min_return,unified_fds_no_min_return_oos_refit,candidate_fit,UNFROZEN_BACK_DIAGNOSTIC,0.036027,...,None,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN
14659,2026-06-26,30,Back,vrp_final_corsi_signal_panel_v1,recomputed_unified_fds_final_signal_001,unified_fds_no_min_return,unified_fds_no_min_return_oos_refit,candidate_fit,UNFROZEN_BACK_DIAGNOSTIC,0.037348,...,None,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN
14668,2026-06-29,30,Back,vrp_final_corsi_signal_panel_v1,recomputed_unified_fds_final_signal_001,unified_fds_no_min_return,unified_fds_no_min_return_oos_refit,candidate_fit,UNFROZEN_BACK_DIAGNOSTIC,0.031021,...,None,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN
14677,2026-06-30,30,Back,vrp_final_corsi_signal_panel_v1,recomputed_unified_fds_final_signal_001,unified_fds_no_min_return,unified_fds_no_min_return_oos_refit,candidate_fit,UNFROZEN_BACK_DIAGNOSTIC,0.027246,...,None,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN
14686,2026-07-01,30,Back,vrp_final_corsi_signal_panel_v1,recomputed_unified_fds_final_signal_001,unified_fds_no_min_return,unified_fds_no_min_return_oos_refit,candidate_fit,UNFROZEN_BACK_DIAGNOSTIC,0.026883,...,None,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN
14695,2026-07-06,30,None,None,None,None,None,None,None,0.024585,...,None,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN
14704,2026-07-07,30,None,None,None,None,None,None,None,0.026037,...,None,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN
14713,2026-07-08,30,None,None,None,None,None,None,None,0.028086,...,None,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN



Saved column inventory: C:\Users\patri\vrp_project\data\audit\call_sleeve_research\01B_call_sleeve_30d_signal_column_inventory_20260709_204205.csv


In [3]:
# Cell 1C — Build 30D call-sleeve signal input panel
# Purpose:
#   Build the clean 30D-only input panel needed to replicate the current Excel short-call sleeve.
#
# Scope:
#   - 30D only
#   - SPY option trade construction inputs
#   - Existing SPX-derived VIX-style vol / Corsi / VRP signal inputs
#   - Signal only, no P&L yet
#   - No RV21D filter
#
# Strategy being replicated:
#   - Sell 1 SD call
#   - Buy 3 SD call
#   - SD = VIX_style_vol_pct / 100 * sqrt(1/12)
#   - Signal:
#       z_3m > 0.1
#       z_1y > 0.1
#       rsi14 > 58

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FINAL_SIGNAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vrp_final_signal"
    / "vrp_final_corsi_signal_base_panel_v1.parquet"
)

OUTPUT_PANEL_PATH = PROCESSED_DIR / "call_sleeve_30d_signal_input_panel_v1.parquet"

print("=" * 100)
print("Cell 1C — Build 30D call-sleeve signal input panel")
print("=" * 100)
print(f"Input:   {FINAL_SIGNAL_PATH}")
print(f"Output:  {OUTPUT_PANEL_PATH}")
print()

# --------------------------------------------------------------------------------------
# Parameters
# --------------------------------------------------------------------------------------

TARGET_TENOR = 30

Z_3M_THRESHOLD = 0.1
Z_1Y_THRESHOLD = 0.1
RSI_THRESHOLD = 58.0

SQRT_T = np.sqrt(1 / 12)

# --------------------------------------------------------------------------------------
# Load and validate source
# --------------------------------------------------------------------------------------

df = pd.read_parquet(FINAL_SIGNAL_PATH)

required_cols = [
    "trade_date",
    "tenor",
    "spy_close",
    "implied_variance_final",
    "vix_style_vol_final",
    "forecast_variance_final",
    "forecast_vol_final",
    "model_vrp_log_final",
    "z_3m_final",
    "z_1y_final",
    "rsi14_final",
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns from final signal panel: {missing_cols}")

# --------------------------------------------------------------------------------------
# Build 30D panel
# --------------------------------------------------------------------------------------

panel = df.loc[df["tenor"] == TARGET_TENOR, required_cols].copy()

panel = panel.rename(
    columns={
        "trade_date": "trade_date",
        "tenor": "tenor",
        "spy_close": "spy_close",
        "implied_variance_final": "implied_variance",
        "vix_style_vol_final": "vix_style_vol_pct",
        "forecast_variance_final": "forecast_variance",
        "forecast_vol_final": "forecast_vol_pct",
        "model_vrp_log_final": "model_vrp_log",
        "z_3m_final": "model_vrp_z_3m",
        "z_1y_final": "model_vrp_z_1y",
        "rsi14_final": "rsi14",
    }
)

panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce")
panel = panel.sort_values("trade_date").reset_index(drop=True)

# --------------------------------------------------------------------------------------
# Guardrails
# --------------------------------------------------------------------------------------

duplicate_dates = panel["trade_date"].duplicated().sum()
if duplicate_dates > 0:
    raise ValueError(f"30D panel has duplicate trade_date rows: {duplicate_dates:,}")

if panel.empty:
    raise ValueError("30D panel is empty.")

vix_median = panel["vix_style_vol_pct"].dropna().median()

# We expect VIX-style vol as a percent, e.g. 16.5, not 0.165.
if pd.notna(vix_median) and vix_median <= 1.0:
    raise ValueError(
        "vix_style_vol_pct appears to be in decimal format, not percent format. "
        f"Median value is {vix_median:.6f}. Expected values like 12, 18, 25."
    )

# --------------------------------------------------------------------------------------
# Signal logic
# --------------------------------------------------------------------------------------

panel["call_signal_z_3m_pass"] = panel["model_vrp_z_3m"] > Z_3M_THRESHOLD
panel["call_signal_z_1y_pass"] = panel["model_vrp_z_1y"] > Z_1Y_THRESHOLD
panel["call_signal_rsi_pass"] = panel["rsi14"] > RSI_THRESHOLD

panel["call_signal_pass"] = (
    panel["call_signal_z_3m_pass"]
    & panel["call_signal_z_1y_pass"]
    & panel["call_signal_rsi_pass"]
)

# --------------------------------------------------------------------------------------
# Strike construction inputs
# --------------------------------------------------------------------------------------

panel["sqrt_t"] = SQRT_T
panel["sd_move_decimal"] = (panel["vix_style_vol_pct"] / 100.0) * panel["sqrt_t"]

panel["short_call_1sd_raw"] = panel["spy_close"] * (1.0 + panel["sd_move_decimal"])
panel["long_call_3sd_raw"] = panel["spy_close"] * (1.0 + 3.0 * panel["sd_move_decimal"])

panel["call_spread_width_raw"] = panel["long_call_3sd_raw"] - panel["short_call_1sd_raw"]

# --------------------------------------------------------------------------------------
# Missingness audit
# --------------------------------------------------------------------------------------

core_signal_cols = [
    "spy_close",
    "vix_style_vol_pct",
    "implied_variance",
    "forecast_variance",
    "forecast_vol_pct",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "rsi14",
]

missing_summary = (
    panel[core_signal_cols]
    .isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)

missing_summary["missing_pct"] = missing_summary["missing_count"] / len(panel)

# --------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

audit_csv_path = AUDIT_DIR / f"01C_call_sleeve_30d_signal_input_panel_{timestamp}.csv"
missing_csv_path = AUDIT_DIR / f"01C_call_sleeve_30d_missing_summary_{timestamp}.csv"
manifest_path = AUDIT_DIR / f"01C_call_sleeve_30d_signal_input_manifest_{timestamp}.json"

panel.to_parquet(OUTPUT_PANEL_PATH, index=False)
panel.to_csv(audit_csv_path, index=False)
missing_summary.to_csv(missing_csv_path, index=False)

signal_count = int(panel["call_signal_pass"].sum())
eligible_count = int(panel[core_signal_cols].notna().all(axis=1).sum())
total_count = int(len(panel))

manifest = {
    "cell": "1C",
    "description": "30D call-sleeve signal input panel",
    "created_at": timestamp,
    "input_path": str(FINAL_SIGNAL_PATH),
    "output_panel_path": str(OUTPUT_PANEL_PATH),
    "audit_csv_path": str(audit_csv_path),
    "missing_csv_path": str(missing_csv_path),
    "target_tenor": TARGET_TENOR,
    "z_3m_threshold": Z_3M_THRESHOLD,
    "z_1y_threshold": Z_1Y_THRESHOLD,
    "rsi_threshold": RSI_THRESHOLD,
    "sqrt_t": SQRT_T,
    "total_rows": total_count,
    "eligible_rows_all_core_inputs_present": eligible_count,
    "signal_count": signal_count,
    "signal_frequency_total_rows": signal_count / total_count if total_count else None,
    "signal_frequency_eligible_rows": signal_count / eligible_count if eligible_count else None,
    "date_min": str(panel["trade_date"].min().date()),
    "date_max": str(panel["trade_date"].max().date()),
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# --------------------------------------------------------------------------------------
# Console output
# --------------------------------------------------------------------------------------

print("Built 30D call-sleeve signal input panel")
print(f"Rows:                       {total_count:,}")
print(f"Date range:                 {panel['trade_date'].min().date()} to {panel['trade_date'].max().date()}")
print(f"Tenors:                     {sorted(panel['tenor'].dropna().unique().tolist())}")
print(f"Duplicate dates:            {duplicate_dates:,}")
print()
print("Signal thresholds:")
print(f"  3m VRP z-score >           {Z_3M_THRESHOLD}")
print(f"  1y VRP z-score >           {Z_1Y_THRESHOLD}")
print(f"  RSI14 >                    {RSI_THRESHOLD}")
print()
print("Signal frequency:")
print(f"  Signal count:              {signal_count:,}")
print(f"  Total rows:                {total_count:,}")
print(f"  Eligible rows:             {eligible_count:,}")
print(f"  Frequency / total rows:    {signal_count / total_count:.2%}")
print(f"  Frequency / eligible rows: {signal_count / eligible_count:.2%}" if eligible_count else "  Frequency / eligible rows: n/a")
print()
print("VIX-style vol sanity:")
print(f"  Median vix_style_vol_pct:  {vix_median:.4f}")
print(f"  Min vix_style_vol_pct:     {panel['vix_style_vol_pct'].min():.4f}")
print(f"  Max vix_style_vol_pct:     {panel['vix_style_vol_pct'].max():.4f}")
print()
print("Saved:")
print(f"  Panel parquet:             {OUTPUT_PANEL_PATH}")
print(f"  Audit CSV:                 {audit_csv_path}")
print(f"  Missing summary CSV:       {missing_csv_path}")
print(f"  Manifest JSON:             {manifest_path}")
print()

print("=" * 100)
print("Missing summary")
print("=" * 100)
display(missing_summary)

print()
print("=" * 100)
print("Latest 10 rows")
print("=" * 100)
display(panel.tail(10))

print()
print("=" * 100)
print("Latest signal rows")
print("=" * 100)
display(panel.loc[panel["call_signal_pass"]].tail(10))

Cell 1C — Build 30D call-sleeve signal input panel
Input:   C:\Users\patri\vrp_project\data\processed\vrp_final_signal\vrp_final_corsi_signal_base_panel_v1.parquet
Output:  C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_30d_signal_input_panel_v1.parquet

Built 30D call-sleeve signal input panel
Rows:                       1,635
Date range:                 2020-01-02 to 2026-07-08
Tenors:                     [30]
Duplicate dates:            0

Signal thresholds:
  3m VRP z-score >           0.1
  1y VRP z-score >           0.1
  RSI14 >                    58.0

Signal frequency:
  Signal count:              172
  Total rows:                1,635
  Eligible rows:             1,380
  Frequency / total rows:    10.52%
  Frequency / eligible rows: 12.46%

VIX-style vol sanity:
  Median vix_style_vol_pct:  18.9504
  Min vix_style_vol_pct:     11.8712
  Max vix_style_vol_pct:     80.4147

Saved:
  Panel parquet:             C:\Users\patri\vrp_project\data\processed

,column,missing_count,missing_pct
0,spy_close,3,0.001835
1,vix_style_vol_pct,0,0.000000
2,implied_variance,0,0.000000
3,forecast_variance,0,0.000000
4,forecast_vol_pct,0,0.000000
5,model_vrp_log,0,0.000000
6,model_vrp_z_3m,63,0.038532
7,model_vrp_z_1y,252,0.154128
8,rsi14,0,0.000000



Latest 10 rows


,trade_date,tenor,spy_close,implied_variance,vix_style_vol_pct,forecast_variance,forecast_vol_pct,model_vrp_log,model_vrp_z_3m,model_vrp_z_1y,rsi14,call_signal_z_3m_pass,call_signal_z_1y_pass,call_signal_rsi_pass,call_signal_pass,sqrt_t,sd_move_decimal,short_call_1sd_raw,long_call_3sd_raw,call_spread_width_raw
1625,2026-06-23,30,733.58,0.038364,19.586777,0.025991,16.121740,0.389372,-0.860850,-0.801066,46.752076,False,False,False,False,0.288675,0.056542,775.058194,858.014583,82.956389
1626,2026-06-24,30,733.24,0.038008,19.495519,0.025876,16.086085,0.384460,-0.850012,-0.816000,46.342083,False,False,False,False,0.288675,0.056279,774.505805,857.037415,82.531610
1627,2026-06-25,30,734.30,0.036027,18.980878,0.024203,15.557217,0.397815,-0.777098,-0.761211,46.297996,False,False,False,False,0.288675,0.054793,774.534556,855.003667,80.469112
1628,2026-06-26,30,728.99,0.037348,19.325534,0.022854,15.117640,0.491130,-0.430644,-0.410699,46.073621,False,False,False,False,0.288675,0.055788,769.658902,850.996707,81.337805
1629,2026-06-29,30,741.00,0.031021,17.612728,0.022847,15.115282,0.305831,-1.053461,-1.112655,52.276135,False,False,False,False,0.288675,0.050844,778.675082,854.025246,75.350164
1630,2026-06-30,30,746.77,0.027246,16.506249,0.019858,14.091786,0.316294,-0.990292,-1.072904,55.993541,False,False,False,False,0.288675,0.047649,782.353169,853.519508,71.166339
1631,2026-07-01,30,745.76,0.026883,16.396160,0.019964,14.129525,0.297561,-1.029999,-1.140252,54.736742,False,False,False,False,0.288675,0.047332,781.058042,851.654125,70.596084
1632,2026-07-06,30,NaN,0.024585,15.679479,0.018556,13.621967,0.281338,-0.841433,-1.044251,52.110740,False,False,False,False,0.288675,0.045263,NaN,NaN,NaN
1633,2026-07-07,30,NaN,0.026037,16.136136,0.019017,13.790318,0.314189,-0.231832,-0.946925,46.359688,False,False,False,False,0.288675,0.046581,NaN,NaN,NaN
1634,2026-07-08,30,NaN,0.028086,16.758938,0.015706,12.532342,0.581238,0.723058,-0.056647,47.044100,True,False,False,False,0.288675,0.048379,NaN,NaN,NaN



Latest signal rows


,trade_date,tenor,spy_close,implied_variance,vix_style_vol_pct,forecast_variance,forecast_vol_pct,model_vrp_log,model_vrp_z_3m,model_vrp_z_1y,rsi14,call_signal_z_3m_pass,call_signal_z_1y_pass,call_signal_rsi_pass,call_signal_pass,sqrt_t,sd_move_decimal,short_call_1sd_raw,long_call_3sd_raw,call_spread_width_raw
1584,2026-04-23,30,708.45,0.036324,19.058987,0.016664,12.908977,0.779231,0.229510,0.738047,67.642796,True,True,True,True,0.288675,0.055019,747.427896,825.383688,77.955792
1586,2026-04-27,30,715.17,0.032904,18.139452,0.015246,12.347463,0.769277,0.120360,0.687981,70.448817,True,True,True,True,0.288675,0.052364,752.619224,827.517673,74.898448
1587,2026-04-28,30,711.69,0.031664,17.794353,0.013596,11.660093,0.845418,0.412331,0.978758,66.880743,True,True,True,True,0.288675,0.051368,748.248001,821.364003,73.116002
1588,2026-04-29,30,711.58,0.033646,18.342717,0.013346,11.552368,0.924684,0.736886,1.279415,66.585933,True,True,True,True,0.288675,0.052951,749.258774,824.616323,75.357549
1591,2026-05-04,30,718.01,0.032887,18.134688,0.014260,11.941425,0.835626,0.291728,0.931680,67.863337,True,True,True,True,0.288675,0.052350,755.598065,830.774195,75.176130
1592,2026-05-05,30,723.77,0.030414,17.439553,0.012826,11.325363,0.863392,0.393868,1.042759,70.785030,True,True,True,True,0.288675,0.050344,760.207226,833.081677,72.874451
1596,2026-05-11,30,739.30,0.033714,18.361465,0.013259,11.514990,0.933209,0.687041,1.316585,75.115221,True,True,True,True,0.288675,0.053005,778.486584,856.859751,78.373168
1597,2026-05-12,30,738.18,0.032112,17.919832,0.012990,11.397203,0.905080,0.526444,1.201445,73.683428,True,True,True,True,0.288675,0.051730,776.366124,852.738372,76.372248
1598,2026-05-13,30,742.31,0.031960,17.877288,0.012365,11.119874,0.949594,0.732757,1.367448,75.514938,True,True,True,True,0.288675,0.051607,780.618604,857.235813,76.617209
1599,2026-05-14,30,748.17,0.029989,17.317399,0.012017,10.962406,0.914480,0.522952,1.226067,77.713863,True,True,True,True,0.288675,0.049991,785.571786,860.375358,74.803572


In [4]:
# Cell 1D — Diagnose call signal frequency versus Excel reference
# Purpose:
#   Diagnose why the Python 30D call signal frequency is below the Excel reference.
#
# Current Python result from Cell 1C:
#   ~12.5% of eligible rows
#
# Excel reference:
#   ~21% frequency
#
# Scope:
#   - Diagnostics only
#   - No payoff
#   - No parameter tuning decision
#   - No production changes

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FINAL_SIGNAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vrp_final_signal"
    / "vrp_final_corsi_signal_base_panel_v1.parquet"
)

print("=" * 100)
print("Cell 1D — Diagnose 30D call signal frequency")
print("=" * 100)
print(f"Input: {FINAL_SIGNAL_PATH}")
print()

# --------------------------------------------------------------------------------------
# Parameters
# --------------------------------------------------------------------------------------

TARGET_TENOR = 30

Z_3M_THRESHOLD = 0.1
Z_1Y_THRESHOLD = 0.1
RSI_THRESHOLD = 58.0

EXCEL_REFERENCE_FREQ = 0.21

# --------------------------------------------------------------------------------------
# Load 30D source panel
# --------------------------------------------------------------------------------------

df = pd.read_parquet(FINAL_SIGNAL_PATH)

required_cols = [
    "trade_date",
    "tenor",
    "spy_close",
    "vix_style_vol_final",
    "model_vrp_log_final",
    "z_3m_final",
    "z_1y_final",
    "source_model_vrp_log",
    "source_model_vrp_z_3m",
    "source_model_vrp_z_1y",
    "rsi14_final",
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

p = df.loc[df["tenor"] == TARGET_TENOR, required_cols].copy()
p["trade_date"] = pd.to_datetime(p["trade_date"], errors="coerce")
p = p.sort_values("trade_date").reset_index(drop=True)
p["year"] = p["trade_date"].dt.year

print("Loaded 30D panel")
print(f"Rows:        {len(p):,}")
print(f"Date range:  {p['trade_date'].min().date()} to {p['trade_date'].max().date()}")
print(f"Duplicate dates: {p['trade_date'].duplicated().sum():,}")
print()

# --------------------------------------------------------------------------------------
# Signal variants
# --------------------------------------------------------------------------------------

def signal_summary(
    data: pd.DataFrame,
    z3_col: str,
    z1_col: str,
    rsi_col: str = "rsi14_final",
    require_spy_close: bool = False,
    label: str = "",
) -> dict:
    needed = [z3_col, z1_col, rsi_col]
    if require_spy_close:
        needed.append("spy_close")
    
    eligible = data[needed].notna().all(axis=1)
    
    z3_pass = data[z3_col] > Z_3M_THRESHOLD
    z1_pass = data[z1_col] > Z_1Y_THRESHOLD
    rsi_pass = data[rsi_col] > RSI_THRESHOLD
    
    signal = eligible & z3_pass & z1_pass & rsi_pass
    
    return {
        "variant": label,
        "z3_col": z3_col,
        "z1_col": z1_col,
        "require_spy_close": require_spy_close,
        "total_rows": int(len(data)),
        "eligible_rows": int(eligible.sum()),
        "z3_pass_count": int((eligible & z3_pass).sum()),
        "z1_pass_count": int((eligible & z1_pass).sum()),
        "rsi_pass_count": int((eligible & rsi_pass).sum()),
        "z3_z1_pass_count": int((eligible & z3_pass & z1_pass).sum()),
        "signal_count": int(signal.sum()),
        "eligible_frequency": float(signal.sum() / eligible.sum()) if eligible.sum() else np.nan,
        "total_frequency": float(signal.sum() / len(data)) if len(data) else np.nan,
        "excel_reference_frequency": EXCEL_REFERENCE_FREQ,
        "gap_vs_excel_reference": float((signal.sum() / eligible.sum()) - EXCEL_REFERENCE_FREQ) if eligible.sum() else np.nan,
    }

variant_rows = []

variant_rows.append(
    signal_summary(
        p,
        z3_col="z_3m_final",
        z1_col="z_1y_final",
        require_spy_close=False,
        label="final_zscores_signal_inputs_only",
    )
)

variant_rows.append(
    signal_summary(
        p,
        z3_col="z_3m_final",
        z1_col="z_1y_final",
        require_spy_close=True,
        label="final_zscores_require_spy_close",
    )
)

variant_rows.append(
    signal_summary(
        p,
        z3_col="source_model_vrp_z_3m",
        z1_col="source_model_vrp_z_1y",
        require_spy_close=False,
        label="source_zscores_signal_inputs_only",
    )
)

variant_rows.append(
    signal_summary(
        p,
        z3_col="source_model_vrp_z_3m",
        z1_col="source_model_vrp_z_1y",
        require_spy_close=True,
        label="source_zscores_require_spy_close",
    )
)

variant_summary = pd.DataFrame(variant_rows)

# --------------------------------------------------------------------------------------
# Component pass rates for final and source variants
# --------------------------------------------------------------------------------------

component_rows = []

for variant_name, z3_col, z1_col in [
    ("final_zscores", "z_3m_final", "z_1y_final"),
    ("source_zscores", "source_model_vrp_z_3m", "source_model_vrp_z_1y"),
]:
    eligible = p[[z3_col, z1_col, "rsi14_final"]].notna().all(axis=1)
    z3_pass = p[z3_col] > Z_3M_THRESHOLD
    z1_pass = p[z1_col] > Z_1Y_THRESHOLD
    rsi_pass = p["rsi14_final"] > RSI_THRESHOLD
    
    checks = {
        "z_3m_pass": z3_pass,
        "z_1y_pass": z1_pass,
        "rsi_pass": rsi_pass,
        "z_3m_and_z_1y_pass": z3_pass & z1_pass,
        "all_three_pass": z3_pass & z1_pass & rsi_pass,
    }
    
    for check_name, mask in checks.items():
        component_rows.append({
            "variant": variant_name,
            "check": check_name,
            "eligible_rows": int(eligible.sum()),
            "pass_count": int((eligible & mask).sum()),
            "pass_rate": float((eligible & mask).sum() / eligible.sum()) if eligible.sum() else np.nan,
        })

component_summary = pd.DataFrame(component_rows)

# --------------------------------------------------------------------------------------
# Frequency by year
# --------------------------------------------------------------------------------------

year_rows = []

for variant_name, z3_col, z1_col in [
    ("final_zscores", "z_3m_final", "z_1y_final"),
    ("source_zscores", "source_model_vrp_z_3m", "source_model_vrp_z_1y"),
]:
    for year, g in p.groupby("year"):
        eligible = g[[z3_col, z1_col, "rsi14_final"]].notna().all(axis=1)
        signal = (
            eligible
            & (g[z3_col] > Z_3M_THRESHOLD)
            & (g[z1_col] > Z_1Y_THRESHOLD)
            & (g["rsi14_final"] > RSI_THRESHOLD)
        )
        
        year_rows.append({
            "variant": variant_name,
            "year": int(year),
            "total_rows": int(len(g)),
            "eligible_rows": int(eligible.sum()),
            "signal_count": int(signal.sum()),
            "eligible_frequency": float(signal.sum() / eligible.sum()) if eligible.sum() else np.nan,
            "total_frequency": float(signal.sum() / len(g)) if len(g) else np.nan,
        })

year_summary = pd.DataFrame(year_rows)

# --------------------------------------------------------------------------------------
# Final vs source z-score comparison
# --------------------------------------------------------------------------------------

comparison = p.copy()

comparison["final_signal"] = (
    comparison[["z_3m_final", "z_1y_final", "rsi14_final"]].notna().all(axis=1)
    & (comparison["z_3m_final"] > Z_3M_THRESHOLD)
    & (comparison["z_1y_final"] > Z_1Y_THRESHOLD)
    & (comparison["rsi14_final"] > RSI_THRESHOLD)
)

comparison["source_signal"] = (
    comparison[["source_model_vrp_z_3m", "source_model_vrp_z_1y", "rsi14_final"]].notna().all(axis=1)
    & (comparison["source_model_vrp_z_3m"] > Z_3M_THRESHOLD)
    & (comparison["source_model_vrp_z_1y"] > Z_1Y_THRESHOLD)
    & (comparison["rsi14_final"] > RSI_THRESHOLD)
)

comparison["source_pass_final_fail"] = comparison["source_signal"] & ~comparison["final_signal"]
comparison["final_pass_source_fail"] = comparison["final_signal"] & ~comparison["source_signal"]

comparison["z_3m_source_minus_final"] = comparison["source_model_vrp_z_3m"] - comparison["z_3m_final"]
comparison["z_1y_source_minus_final"] = comparison["source_model_vrp_z_1y"] - comparison["z_1y_final"]
comparison["vrp_log_source_minus_final"] = comparison["source_model_vrp_log"] - comparison["model_vrp_log_final"]

disagreement_summary = pd.DataFrame([
    {
        "metric": "source_signal_true_final_signal_false",
        "count": int(comparison["source_pass_final_fail"].sum()),
    },
    {
        "metric": "final_signal_true_source_signal_false",
        "count": int(comparison["final_pass_source_fail"].sum()),
    },
    {
        "metric": "both_signal_true",
        "count": int((comparison["source_signal"] & comparison["final_signal"]).sum()),
    },
    {
        "metric": "neither_signal_true",
        "count": int((~comparison["source_signal"] & ~comparison["final_signal"]).sum()),
    },
])

zscore_diff_summary = comparison[
    [
        "z_3m_source_minus_final",
        "z_1y_source_minus_final",
        "vrp_log_source_minus_final",
    ]
].describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).T.reset_index().rename(columns={"index": "field"})

# --------------------------------------------------------------------------------------
# Near-threshold diagnostics
# --------------------------------------------------------------------------------------

near = comparison.copy()

near["final_z3_distance_to_threshold"] = near["z_3m_final"] - Z_3M_THRESHOLD
near["final_z1_distance_to_threshold"] = near["z_1y_final"] - Z_1Y_THRESHOLD
near["rsi_distance_to_threshold"] = near["rsi14_final"] - RSI_THRESHOLD

near_fail_rows = near.loc[
    near[["z_3m_final", "z_1y_final", "rsi14_final"]].notna().all(axis=1)
    & ~near["final_signal"]
].copy()

near_fail_rows["max_failed_distance"] = np.nan

failed_distances = []
for _, row in near_fail_rows.iterrows():
    distances = []
    if row["z_3m_final"] <= Z_3M_THRESHOLD:
        distances.append(row["final_z3_distance_to_threshold"])
    if row["z_1y_final"] <= Z_1Y_THRESHOLD:
        distances.append(row["final_z1_distance_to_threshold"])
    if row["rsi14_final"] <= RSI_THRESHOLD:
        distances.append(row["rsi_distance_to_threshold"])
    failed_distances.append(max(distances) if distances else np.nan)

near_fail_rows["closest_failed_filter_distance"] = failed_distances
near_fail_rows = near_fail_rows.sort_values("closest_failed_filter_distance", ascending=False)

# --------------------------------------------------------------------------------------
# Save audit outputs
# --------------------------------------------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

variant_summary_path = AUDIT_DIR / f"01D_call_sleeve_frequency_variant_summary_{timestamp}.csv"
component_summary_path = AUDIT_DIR / f"01D_call_sleeve_component_pass_summary_{timestamp}.csv"
year_summary_path = AUDIT_DIR / f"01D_call_sleeve_frequency_by_year_{timestamp}.csv"
disagreement_path = AUDIT_DIR / f"01D_call_sleeve_final_vs_source_disagreements_{timestamp}.csv"
zscore_diff_path = AUDIT_DIR / f"01D_call_sleeve_zscore_diff_summary_{timestamp}.csv"
near_fail_path = AUDIT_DIR / f"01D_call_sleeve_near_threshold_failures_{timestamp}.csv"
manifest_path = AUDIT_DIR / f"01D_call_sleeve_frequency_diagnostic_manifest_{timestamp}.json"

variant_summary.to_csv(variant_summary_path, index=False)
component_summary.to_csv(component_summary_path, index=False)
year_summary.to_csv(year_summary_path, index=False)
comparison.to_csv(disagreement_path, index=False)
zscore_diff_summary.to_csv(zscore_diff_path, index=False)
near_fail_rows.to_csv(near_fail_path, index=False)

manifest = {
    "cell": "1D",
    "description": "30D call signal frequency diagnostics versus Excel reference",
    "created_at": timestamp,
    "input_path": str(FINAL_SIGNAL_PATH),
    "target_tenor": TARGET_TENOR,
    "z_3m_threshold": Z_3M_THRESHOLD,
    "z_1y_threshold": Z_1Y_THRESHOLD,
    "rsi_threshold": RSI_THRESHOLD,
    "excel_reference_frequency": EXCEL_REFERENCE_FREQ,
    "variant_summary_path": str(variant_summary_path),
    "component_summary_path": str(component_summary_path),
    "year_summary_path": str(year_summary_path),
    "disagreement_path": str(disagreement_path),
    "zscore_diff_path": str(zscore_diff_path),
    "near_fail_path": str(near_fail_path),
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# --------------------------------------------------------------------------------------
# Console output
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Variant signal frequency summary")
print("=" * 100)
display(
    variant_summary[
        [
            "variant",
            "total_rows",
            "eligible_rows",
            "signal_count",
            "eligible_frequency",
            "total_frequency",
            "gap_vs_excel_reference",
        ]
    ]
)

print()
print("=" * 100)
print("Component pass summary")
print("=" * 100)
display(component_summary)

print()
print("=" * 100)
print("Frequency by year")
print("=" * 100)
display(year_summary)

print()
print("=" * 100)
print("Final vs source signal disagreement summary")
print("=" * 100)
display(disagreement_summary)

print()
print("=" * 100)
print("Z-score / VRP source-minus-final differences")
print("=" * 100)
display(zscore_diff_summary)

print()
print("=" * 100)
print("Latest source-pass / final-fail rows")
print("=" * 100)
display(
    comparison.loc[comparison["source_pass_final_fail"]]
    .sort_values("trade_date")
    .tail(20)
    [
        [
            "trade_date",
            "spy_close",
            "model_vrp_log_final",
            "source_model_vrp_log",
            "z_3m_final",
            "source_model_vrp_z_3m",
            "z_1y_final",
            "source_model_vrp_z_1y",
            "rsi14_final",
            "final_signal",
            "source_signal",
        ]
    ]
)

print()
print("=" * 100)
print("Closest final-signal failures to passing")
print("=" * 100)
display(
    near_fail_rows.tail(0) if near_fail_rows.empty else near_fail_rows.head(20)[
        [
            "trade_date",
            "spy_close",
            "z_3m_final",
            "z_1y_final",
            "rsi14_final",
            "final_z3_distance_to_threshold",
            "final_z1_distance_to_threshold",
            "rsi_distance_to_threshold",
            "closest_failed_filter_distance",
        ]
    ]
)

print()
print("Saved:")
print(f"  Variant summary:         {variant_summary_path}")
print(f"  Component summary:       {component_summary_path}")
print(f"  Frequency by year:       {year_summary_path}")
print(f"  Disagreement detail:     {disagreement_path}")
print(f"  Z-score diff summary:    {zscore_diff_path}")
print(f"  Near-threshold failures: {near_fail_path}")
print(f"  Manifest:                {manifest_path}")

Cell 1D — Diagnose 30D call signal frequency
Input: C:\Users\patri\vrp_project\data\processed\vrp_final_signal\vrp_final_corsi_signal_base_panel_v1.parquet

Loaded 30D panel
Rows:        1,635
Date range:  2020-01-02 to 2026-07-08
Duplicate dates: 0

Variant signal frequency summary


,variant,total_rows,eligible_rows,signal_count,eligible_frequency,total_frequency,gap_vs_excel_reference
0,final_zscores_signal_inputs_only,1635,1383,172,0.124367,0.105199,-0.085633
1,final_zscores_require_spy_close,1635,1380,172,0.124638,0.105199,-0.085362
2,source_zscores_signal_inputs_only,1635,1380,184,0.133333,0.112538,-0.076667
3,source_zscores_require_spy_close,1635,1380,184,0.133333,0.112538,-0.076667



Component pass summary


,variant,check,eligible_rows,pass_count,pass_rate
0,final_zscores,z_3m_pass,1383,578,0.417932
1,final_zscores,z_1y_pass,1383,539,0.389732
2,final_zscores,rsi_pass,1383,651,0.470716
3,final_zscores,z_3m_and_z_1y_pass,1383,432,0.312364
4,final_zscores,all_three_pass,1383,172,0.124367
5,source_zscores,z_3m_pass,1380,572,0.414493
6,source_zscores,z_1y_pass,1380,514,0.372464
7,source_zscores,rsi_pass,1380,651,0.471739
8,source_zscores,z_3m_and_z_1y_pass,1380,440,0.318841
9,source_zscores,all_three_pass,1380,184,0.133333



Frequency by year


...


Latest source-pass / final-fail rows


,trade_date,spy_close,model_vrp_log_final,source_model_vrp_log,z_3m_final,source_model_vrp_z_3m,z_1y_final,source_model_vrp_z_1y,rsi14_final,final_signal,source_signal
1129,2024-06-28,544.22,0.198600,0.393304,0.032418,0.851827,0.225288,1.028950,65.133073,False,True
1131,2024-07-02,549.01,0.198695,0.373891,0.053085,0.732686,0.216306,0.911766,71.094162,False,True
1223,2024-11-11,598.76,0.290800,0.509972,-0.600145,0.186791,0.180972,0.822459,69.163221,False,True
1233,2024-11-25,597.53,0.412873,0.700836,-0.178498,0.632637,0.661561,1.446549,63.132771,False,True
1234,2024-11-26,600.65,0.331668,0.561148,-0.496263,0.184287,0.319429,0.952716,66.090661,False,True
1354,2025-05-22,583.09,0.302236,0.487667,-0.368496,0.187995,-0.147054,0.449963,59.325106,False,True
1376,2025-06-25,607.12,0.333426,0.537763,-0.272969,0.428092,-0.119987,0.568591,65.107321,False,True
1378,2025-06-27,614.91,0.427188,0.566940,0.049427,0.507830,0.252949,0.665540,70.214829,False,True
1408,2025-08-11,635.92,0.515768,0.552709,-0.027018,0.132674,0.404404,0.457391,60.480987,False,True
1414,2025-08-19,639.81,0.516432,0.591187,-0.141143,0.156173,0.391920,0.572351,58.861319,False,True



Closest final-signal failures to passing


,trade_date,spy_close,z_3m_final,z_1y_final,rsi14_final,final_z3_distance_to_threshold,final_z1_distance_to_threshold,rsi_distance_to_threshold,closest_failed_filter_distance
1467,2025-11-03,683.34,-0.631905,0.099993,62.361586,-0.731905,-0.000007,4.361586,-0.000007
1431,2025-09-12,657.41,0.099780,0.635200,67.239498,-0.000220,0.535200,9.239498,-0.000220
949,2023-10-10,434.54,2.241245,0.098899,50.424651,2.141245,-0.001101,-7.575349,-0.001101
816,2023-03-30,403.70,0.098340,-0.585008,57.930283,-0.001660,-0.685008,-0.069717,-0.001660
1103,2024-05-21,531.36,-0.090522,0.097007,69.580000,-0.190522,-0.002993,11.580000,-0.002993
1421,2025-08-28,648.92,-0.564607,0.091388,64.601660,-0.664607,-0.008612,6.601660,-0.008612
907,2023-08-10,445.91,0.089888,-0.796913,46.496624,-0.010112,-0.896913,-11.503376,-0.010112
640,2022-07-19,392.27,0.089850,0.321261,55.149039,-0.010150,0.221261,-2.850961,-0.010150
914,2023-08-21,439.34,0.088837,-0.812716,40.446319,-0.011163,-0.912716,-17.553681,-0.011163
995,2023-12-14,472.01,0.088615,-0.489111,79.046948,-0.011385,-0.589111,21.046948,-0.011385



Saved:
  Variant summary:         C:\Users\patri\vrp_project\data\audit\call_sleeve_research\01D_call_sleeve_frequency_variant_summary_20260709_204205.csv
  Component summary:       C:\Users\patri\vrp_project\data\audit\call_sleeve_research\01D_call_sleeve_component_pass_summary_20260709_204205.csv
  Frequency by year:       C:\Users\patri\vrp_project\data\audit\call_sleeve_research\01D_call_sleeve_frequency_by_year_20260709_204205.csv
  Disagreement detail:     C:\Users\patri\vrp_project\data\audit\call_sleeve_research\01D_call_sleeve_final_vs_source_disagreements_20260709_204205.csv
  Z-score diff summary:    C:\Users\patri\vrp_project\data\audit\call_sleeve_research\01D_call_sleeve_zscore_diff_summary_20260709_204205.csv
  Near-threshold failures: C:\Users\patri\vrp_project\data\audit\call_sleeve_research\01D_call_sleeve_near_threshold_failures_20260709_204205.csv
  Manifest:                C:\Users\patri\vrp_project\data\audit\call_sleeve_research\01D_call_sleeve_frequency_diagnos

In [5]:
# Cell 1E — Build 30D Excel-style VRP and call signal panel
# Purpose:
#   Replicate the current Excel short-call signal definition:
#
#       VRP = LN(VIX^2 / RV)
#
#   where:
#       VIX = existing SPX-derived 30D VIX-style vol, expressed as percent
#       RV  = 21-trading-day realized variance, expressed through rv21d_vol_pct_final
#
# Scope:
#   - 30D only
#   - SPY option trade construction inputs
#   - Existing SPX-derived VIX-style vol
#   - Existing RV21D realized vol denominator
#   - Existing RSI14
#   - Signal only, no P&L yet
#   - No Corsi forecast denominator for Excel replication
#
# Strategy being replicated:
#   - Sell 1 SD call
#   - Buy 3 SD call
#   - SD = VIX_style_vol_pct / 100 * sqrt(1/12)
#   - Signal:
#       Excel-style 3m VRP z-score > 0.1
#       Excel-style 1y VRP z-score > 0.1
#       RSI14 > 58
#
# Z-score convention:
#   - Current day's VRP is used as x
#   - Rolling mean/std window excludes current day via shift(1)
#   - Rolling std uses sample std, matching Excel STDEV.S behavior

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FINAL_SIGNAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vrp_final_signal"
    / "vrp_final_corsi_signal_base_panel_v1.parquet"
)

OUTPUT_PANEL_PATH = PROCESSED_DIR / "call_sleeve_30d_excel_vrp_signal_panel_v1.parquet"

print("=" * 100)
print("Cell 1E — Build 30D Excel-style VRP call signal panel")
print("=" * 100)
print(f"Input:   {FINAL_SIGNAL_PATH}")
print(f"Output:  {OUTPUT_PANEL_PATH}")
print()

# --------------------------------------------------------------------------------------
# Parameters
# --------------------------------------------------------------------------------------

TARGET_TENOR = 30

Z_3M_WINDOW = 63
Z_1Y_WINDOW = 252

Z_3M_THRESHOLD = 0.1
Z_1Y_THRESHOLD = 0.1
RSI_THRESHOLD = 58.0

SQRT_T = np.sqrt(1 / 12)

# --------------------------------------------------------------------------------------
# Load source
# --------------------------------------------------------------------------------------

df = pd.read_parquet(FINAL_SIGNAL_PATH)

required_cols = [
    "trade_date",
    "tenor",
    "spy_close",
    "vix_style_vol_final",
    "rv21d_vol_pct_final",
    "rsi14_final",
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns from final signal panel: {missing_cols}")

# --------------------------------------------------------------------------------------
# Build 30D panel
# --------------------------------------------------------------------------------------

panel = df.loc[df["tenor"] == TARGET_TENOR, required_cols].copy()

panel = panel.rename(
    columns={
        "trade_date": "trade_date",
        "tenor": "tenor",
        "spy_close": "spy_close",
        "vix_style_vol_final": "vix_style_vol_pct",
        "rv21d_vol_pct_final": "rv21d_vol_pct",
        "rsi14_final": "rsi14",
    }
)

panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce")
panel = panel.sort_values("trade_date").reset_index(drop=True)

# --------------------------------------------------------------------------------------
# Guardrails
# --------------------------------------------------------------------------------------

if panel.empty:
    raise ValueError("30D panel is empty.")

duplicate_dates = panel["trade_date"].duplicated().sum()
if duplicate_dates > 0:
    raise ValueError(f"30D panel has duplicate trade_date rows: {duplicate_dates:,}")

vix_median = panel["vix_style_vol_pct"].dropna().median()
rv_median = panel["rv21d_vol_pct"].dropna().median()

if pd.notna(vix_median) and vix_median <= 1.0:
    raise ValueError(
        "vix_style_vol_pct appears to be in decimal format, not percent format. "
        f"Median value is {vix_median:.6f}. Expected values like 12, 18, 25."
    )

if pd.notna(rv_median) and rv_median <= 1.0:
    raise ValueError(
        "rv21d_vol_pct appears to be in decimal format, not percent format. "
        f"Median value is {rv_median:.6f}. Expected values like 8, 15, 25."
    )

# --------------------------------------------------------------------------------------
# Excel-style VRP calculation
# --------------------------------------------------------------------------------------

panel["vix_variance_decimal"] = (panel["vix_style_vol_pct"] / 100.0) ** 2
panel["rv21d_variance_decimal"] = (panel["rv21d_vol_pct"] / 100.0) ** 2

panel["call_excel_vrp_log"] = np.where(
    (panel["vix_variance_decimal"] > 0)
    & (panel["rv21d_variance_decimal"] > 0)
    & panel["vix_variance_decimal"].notna()
    & panel["rv21d_variance_decimal"].notna(),
    np.log(panel["vix_variance_decimal"] / panel["rv21d_variance_decimal"]),
    np.nan,
)

# Prior-only rolling windows.
# Current day VRP is used in numerator; historical mean/std excludes current day.
panel["call_excel_vrp_3m_mean_prior"] = (
    panel["call_excel_vrp_log"]
    .shift(1)
    .rolling(window=Z_3M_WINDOW, min_periods=Z_3M_WINDOW)
    .mean()
)

panel["call_excel_vrp_3m_std_prior"] = (
    panel["call_excel_vrp_log"]
    .shift(1)
    .rolling(window=Z_3M_WINDOW, min_periods=Z_3M_WINDOW)
    .std(ddof=1)
)

panel["call_excel_vrp_1y_mean_prior"] = (
    panel["call_excel_vrp_log"]
    .shift(1)
    .rolling(window=Z_1Y_WINDOW, min_periods=Z_1Y_WINDOW)
    .mean()
)

panel["call_excel_vrp_1y_std_prior"] = (
    panel["call_excel_vrp_log"]
    .shift(1)
    .rolling(window=Z_1Y_WINDOW, min_periods=Z_1Y_WINDOW)
    .std(ddof=1)
)

panel["call_excel_vrp_z_3m"] = (
    (panel["call_excel_vrp_log"] - panel["call_excel_vrp_3m_mean_prior"])
    / panel["call_excel_vrp_3m_std_prior"]
)

panel["call_excel_vrp_z_1y"] = (
    (panel["call_excel_vrp_log"] - panel["call_excel_vrp_1y_mean_prior"])
    / panel["call_excel_vrp_1y_std_prior"]
)

# --------------------------------------------------------------------------------------
# Signal logic
# --------------------------------------------------------------------------------------

panel["call_signal_z_3m_pass"] = panel["call_excel_vrp_z_3m"] > Z_3M_THRESHOLD
panel["call_signal_z_1y_pass"] = panel["call_excel_vrp_z_1y"] > Z_1Y_THRESHOLD
panel["call_signal_rsi_pass"] = panel["rsi14"] > RSI_THRESHOLD

panel["call_signal_pass"] = (
    panel["call_signal_z_3m_pass"]
    & panel["call_signal_z_1y_pass"]
    & panel["call_signal_rsi_pass"]
)

# --------------------------------------------------------------------------------------
# Strike construction inputs
# --------------------------------------------------------------------------------------

panel["sqrt_t"] = SQRT_T
panel["sd_move_decimal"] = (panel["vix_style_vol_pct"] / 100.0) * panel["sqrt_t"]

panel["short_call_1sd_raw"] = panel["spy_close"] * (1.0 + panel["sd_move_decimal"])
panel["long_call_3sd_raw"] = panel["spy_close"] * (1.0 + 3.0 * panel["sd_move_decimal"])
panel["call_spread_width_raw"] = panel["long_call_3sd_raw"] - panel["short_call_1sd_raw"]

# --------------------------------------------------------------------------------------
# Missingness audit
# --------------------------------------------------------------------------------------

core_signal_cols = [
    "vix_style_vol_pct",
    "rv21d_vol_pct",
    "rsi14",
    "call_excel_vrp_log",
    "call_excel_vrp_z_3m",
    "call_excel_vrp_z_1y",
]

trade_construction_cols = [
    "spy_close",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
    "call_spread_width_raw",
]

missing_summary = (
    panel[core_signal_cols + trade_construction_cols]
    .isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)

missing_summary["missing_pct"] = missing_summary["missing_count"] / len(panel)

eligible_signal_rows = panel[core_signal_cols].notna().all(axis=1)
eligible_trade_rows = panel[core_signal_cols + trade_construction_cols].notna().all(axis=1)

signal_count = int(panel.loc[eligible_signal_rows, "call_signal_pass"].sum())
signal_count_trade_eligible = int(panel.loc[eligible_trade_rows, "call_signal_pass"].sum())

# --------------------------------------------------------------------------------------
# Frequency by year
# --------------------------------------------------------------------------------------

panel["year"] = panel["trade_date"].dt.year

year_summary_rows = []

for year, g in panel.groupby("year"):
    eligible_signal = g[core_signal_cols].notna().all(axis=1)
    eligible_trade = g[core_signal_cols + trade_construction_cols].notna().all(axis=1)
    
    year_summary_rows.append({
        "year": int(year),
        "total_rows": int(len(g)),
        "eligible_signal_rows": int(eligible_signal.sum()),
        "eligible_trade_rows": int(eligible_trade.sum()),
        "signal_count": int(g.loc[eligible_signal, "call_signal_pass"].sum()),
        "signal_count_trade_eligible": int(g.loc[eligible_trade, "call_signal_pass"].sum()),
        "signal_frequency_eligible_signal_rows": (
            float(g.loc[eligible_signal, "call_signal_pass"].sum() / eligible_signal.sum())
            if eligible_signal.sum()
            else np.nan
        ),
        "signal_frequency_eligible_trade_rows": (
            float(g.loc[eligible_trade, "call_signal_pass"].sum() / eligible_trade.sum())
            if eligible_trade.sum()
            else np.nan
        ),
    })

year_summary = pd.DataFrame(year_summary_rows)

# --------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

audit_csv_path = AUDIT_DIR / f"01E_call_sleeve_30d_excel_vrp_signal_panel_{timestamp}.csv"
missing_csv_path = AUDIT_DIR / f"01E_call_sleeve_30d_excel_vrp_missing_summary_{timestamp}.csv"
year_summary_path = AUDIT_DIR / f"01E_call_sleeve_30d_excel_vrp_frequency_by_year_{timestamp}.csv"
manifest_path = AUDIT_DIR / f"01E_call_sleeve_30d_excel_vrp_signal_manifest_{timestamp}.json"

panel.to_parquet(OUTPUT_PANEL_PATH, index=False)
panel.to_csv(audit_csv_path, index=False)
missing_summary.to_csv(missing_csv_path, index=False)
year_summary.to_csv(year_summary_path, index=False)

manifest = {
    "cell": "1E",
    "description": "30D Excel-style VRP call-sleeve signal panel",
    "created_at": timestamp,
    "input_path": str(FINAL_SIGNAL_PATH),
    "output_panel_path": str(OUTPUT_PANEL_PATH),
    "audit_csv_path": str(audit_csv_path),
    "missing_csv_path": str(missing_csv_path),
    "year_summary_path": str(year_summary_path),
    "target_tenor": TARGET_TENOR,
    "vrp_definition": "ln((vix_style_vol_pct/100)^2 / (rv21d_vol_pct/100)^2)",
    "z_3m_window": Z_3M_WINDOW,
    "z_1y_window": Z_1Y_WINDOW,
    "z_window_excludes_current_day": True,
    "z_current_day_value_used_as_observation": True,
    "rolling_std_ddof": 1,
    "z_3m_threshold": Z_3M_THRESHOLD,
    "z_1y_threshold": Z_1Y_THRESHOLD,
    "rsi_threshold": RSI_THRESHOLD,
    "sqrt_t": SQRT_T,
    "total_rows": int(len(panel)),
    "eligible_signal_rows": int(eligible_signal_rows.sum()),
    "eligible_trade_rows": int(eligible_trade_rows.sum()),
    "signal_count": signal_count,
    "signal_count_trade_eligible": signal_count_trade_eligible,
    "signal_frequency_eligible_signal_rows": (
        signal_count / int(eligible_signal_rows.sum()) if int(eligible_signal_rows.sum()) else None
    ),
    "signal_frequency_eligible_trade_rows": (
        signal_count_trade_eligible / int(eligible_trade_rows.sum()) if int(eligible_trade_rows.sum()) else None
    ),
    "date_min": str(panel["trade_date"].min().date()),
    "date_max": str(panel["trade_date"].max().date()),
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# --------------------------------------------------------------------------------------
# Console output
# --------------------------------------------------------------------------------------

print("Built 30D Excel-style VRP call-sleeve signal panel")
print(f"Rows:                              {len(panel):,}")
print(f"Date range:                        {panel['trade_date'].min().date()} to {panel['trade_date'].max().date()}")
print(f"Tenors:                            {sorted(panel['tenor'].dropna().unique().tolist())}")
print(f"Duplicate dates:                   {duplicate_dates:,}")
print()
print("VRP definition:")
print("  call_excel_vrp_log = ln((vix_style_vol_pct / 100)^2 / (rv21d_vol_pct / 100)^2)")
print()
print("Z-score convention:")
print("  Current day VRP used as x")
print("  Rolling mean/std exclude current day via shift(1)")
print("  Rolling std uses sample std, ddof=1")
print()
print("Signal thresholds:")
print(f"  3m Excel VRP z-score >            {Z_3M_THRESHOLD}")
print(f"  1y Excel VRP z-score >            {Z_1Y_THRESHOLD}")
print(f"  RSI14 >                           {RSI_THRESHOLD}")
print()
print("Signal frequency:")
print(f"  Signal count:                     {signal_count:,}")
print(f"  Signal-eligible rows:             {int(eligible_signal_rows.sum()):,}")
print(f"  Trade-eligible rows:              {int(eligible_trade_rows.sum()):,}")
print(f"  Frequency / signal-eligible rows: {signal_count / int(eligible_signal_rows.sum()):.2%}")
print(f"  Frequency / trade-eligible rows:  {signal_count_trade_eligible / int(eligible_trade_rows.sum()):.2%}")
print()
print("Vol sanity:")
print(f"  Median vix_style_vol_pct:         {vix_median:.4f}")
print(f"  Median rv21d_vol_pct:             {rv_median:.4f}")
print(f"  Median call_excel_vrp_log:        {panel['call_excel_vrp_log'].median():.4f}")
print()
print("Saved:")
print(f"  Panel parquet:                    {OUTPUT_PANEL_PATH}")
print(f"  Audit CSV:                        {audit_csv_path}")
print(f"  Missing summary CSV:              {missing_csv_path}")
print(f"  Year summary CSV:                 {year_summary_path}")
print(f"  Manifest JSON:                    {manifest_path}")

print()
print("=" * 100)
print("Missing summary")
print("=" * 100)
display(missing_summary)

print()
print("=" * 100)
print("Frequency by year")
print("=" * 100)
display(year_summary)

print()
print("=" * 100)
print("Latest 10 rows")
print("=" * 100)
display(panel.tail(10))

print()
print("=" * 100)
print("Latest signal rows")
print("=" * 100)
display(panel.loc[panel["call_signal_pass"]].tail(10))

Cell 1E — Build 30D Excel-style VRP call signal panel
Input:   C:\Users\patri\vrp_project\data\processed\vrp_final_signal\vrp_final_corsi_signal_base_panel_v1.parquet
Output:  C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_30d_excel_vrp_signal_panel_v1.parquet

Built 30D Excel-style VRP call-sleeve signal panel
Rows:                              1,635
Date range:                        2020-01-02 to 2026-07-08
Tenors:                            [30]
Duplicate dates:                   0

VRP definition:
  call_excel_vrp_log = ln((vix_style_vol_pct / 100)^2 / (rv21d_vol_pct / 100)^2)

Z-score convention:
  Current day VRP used as x
  Rolling mean/std exclude current day via shift(1)
  Rolling std uses sample std, ddof=1

Signal thresholds:
  3m Excel VRP z-score >            0.1
  1y Excel VRP z-score >            0.1
  RSI14 >                           58.0

Signal frequency:
  Signal count:                     285
  Signal-eligible rows:             1,383
  

,column,missing_count,missing_pct
0,vix_style_vol_pct,0,0.000000
1,rv21d_vol_pct,0,0.000000
2,rsi14,0,0.000000
3,call_excel_vrp_log,0,0.000000
4,call_excel_vrp_z_3m,63,0.038532
5,call_excel_vrp_z_1y,252,0.154128
6,spy_close,3,0.001835
7,short_call_1sd_raw,3,0.001835
8,long_call_3sd_raw,3,0.001835
9,call_spread_width_raw,3,0.001835



Frequency by year


,year,total_rows,eligible_signal_rows,eligible_trade_rows,signal_count,signal_count_trade_eligible,signal_frequency_eligible_signal_rows,signal_frequency_eligible_trade_rows
0,2020,253,1,1,1,1,1.000000,1.000000
1,2021,252,252,252,72,72,0.285714,0.285714
2,2022,251,251,251,0,0,0.000000,0.000000
3,2023,250,250,250,58,58,0.232000,0.232000
4,2024,252,252,252,69,69,0.273810,0.273810
5,2025,250,250,250,62,62,0.248000,0.248000
6,2026,127,127,124,23,23,0.181102,0.185484



Latest 10 rows


,trade_date,tenor,spy_close,vix_style_vol_pct,rv21d_vol_pct,rsi14,vix_variance_decimal,rv21d_variance_decimal,call_excel_vrp_log,call_excel_vrp_3m_mean_prior,...,call_signal_z_3m_pass,call_signal_z_1y_pass,call_signal_rsi_pass,call_signal_pass,sqrt_t,sd_move_decimal,short_call_1sd_raw,long_call_3sd_raw,call_spread_width_raw,year
1625,2026-06-23,30,733.58,19.586777,16.680000,46.752076,0.038364,0.027822,0.321289,0.654276,...,False,False,False,False,0.288675,0.056542,775.058194,858.014583,82.956389,2026
1626,2026-06-24,30,733.24,19.495519,16.599462,46.342083,0.038008,0.027554,0.321629,0.639358,...,False,False,False,False,0.288675,0.056279,774.505805,857.037415,82.531610,2026
1627,2026-06-25,30,734.30,18.980878,16.403886,46.297996,0.036027,0.026909,0.291828,0.623109,...,False,False,False,False,0.288675,0.054793,774.534556,855.003667,80.469112,2026
1628,2026-06-26,30,728.99,19.325534,16.539470,46.073621,0.037348,0.027355,0.311355,0.608010,...,False,False,False,False,0.288675,0.055788,769.658902,850.996707,81.337805,2026
1629,2026-06-29,30,741.00,17.612728,17.505857,52.276135,0.031021,0.030646,0.012173,0.590834,...,False,False,False,False,0.288675,0.050844,778.675082,854.025246,75.350164,2026
1630,2026-06-30,30,746.77,16.506249,17.726787,55.993541,0.027246,0.031424,-0.142676,0.567783,...,False,False,False,False,0.288675,0.047649,782.353169,853.519508,71.166339,2026
1631,2026-07-01,30,745.76,16.396160,17.686351,54.736742,0.026883,0.031281,-0.151492,0.542839,...,False,False,False,False,0.288675,0.047332,781.058042,851.654125,70.596084,2026
1632,2026-07-06,30,NaN,15.679479,17.008710,52.110740,0.024585,0.028930,-0.162745,0.530443,...,False,False,False,False,0.288675,0.045263,NaN,NaN,NaN,2026
1633,2026-07-07,30,NaN,16.136136,17.030961,46.359688,0.026037,0.029005,-0.107943,0.519414,...,False,False,False,False,0.288675,0.046581,NaN,NaN,NaN,2026
1634,2026-07-08,30,NaN,16.758938,14.317996,47.044100,0.028086,0.020501,0.314829,0.509324,...,False,False,False,False,0.288675,0.048379,NaN,NaN,NaN,2026



Latest signal rows


,trade_date,tenor,spy_close,vix_style_vol_pct,rv21d_vol_pct,rsi14,vix_variance_decimal,rv21d_variance_decimal,call_excel_vrp_log,call_excel_vrp_3m_mean_prior,...,call_signal_z_3m_pass,call_signal_z_1y_pass,call_signal_rsi_pass,call_signal_pass,sqrt_t,sd_move_decimal,short_call_1sd_raw,long_call_3sd_raw,call_spread_width_raw,year
1601,2026-05-18,30,738.65,17.715762,10.495996,66.573171,0.031385,0.011017,1.046922,0.818868,...,True,True,True,True,0.288675,0.051141,776.425300,851.975901,75.550601,2026
1602,2026-05-19,30,733.73,18.078614,10.833674,61.450795,0.032684,0.011737,1.024141,0.823574,...,True,True,True,True,0.288675,0.052188,772.022242,848.606725,76.584483,2026
1603,2026-05-20,30,741.25,17.356872,10.787464,65.976250,0.030126,0.011637,0.951207,0.829474,...,True,True,True,True,0.288675,0.050105,778.390311,852.670933,74.280622,2026
1604,2026-05-21,30,742.72,16.726057,10.424796,66.653563,0.027976,0.010868,0.945561,0.828143,...,True,True,True,True,0.288675,0.048284,778.581468,850.304403,71.722935,2026
1605,2026-05-22,30,745.64,16.800819,10.211818,68.140144,0.028227,0.010428,0.995764,0.827501,...,True,True,True,True,0.288675,0.048500,781.803381,854.130144,72.326763,2026
1606,2026-05-26,30,750.59,16.928592,10.146871,70.472169,0.028658,0.010296,1.023677,0.825288,...,True,True,True,True,0.288675,0.048869,787.270308,860.630925,73.360617,2026
1607,2026-05-27,30,750.46,16.321674,10.183614,70.535259,0.026640,0.010371,0.943428,0.827122,...,True,True,True,True,0.288675,0.047117,785.819133,856.537400,70.718267,2026
1610,2026-06-01,30,758.54,15.993564,9.502584,74.510414,0.025579,0.009030,1.041245,0.833081,...,True,True,True,True,0.288675,0.046169,793.561369,863.604106,70.042738,2026
1611,2026-06-02,30,759.57,15.794330,9.511490,75.002578,0.024946,0.009047,1.014301,0.833164,...,True,True,True,True,0.288675,0.045594,794.202066,863.466198,69.264132,2026
1612,2026-06-03,30,754.24,16.049578,9.852886,67.038959,0.025759,0.009708,0.975836,0.830588,...,True,True,True,True,0.288675,0.046331,789.184799,859.074397,69.889598,2026


In [6]:
# Cell 1F — Add 30D held-to-expiration outcome
# Purpose:
#   Validate the current Excel-style call sleeve trade mechanics:
#   - Enter at close
#   - Hold approximately 30 calendar days
#   - Sell 1 SD call
#   - Buy 3 SD call
#   - Check whether the short call expired OTM
#
# Scope:
#   - Validation of trade construction / expiration outcome only
#   - Uses Excel-style VRP signal panel from Cell 1E as the baseline/control
#   - Does NOT calculate full option P&L yet because we do not yet have actual option credits
#   - Does NOT decide final signal model
#
# Exit convention:
#   - exit_target_date = trade_date + 30 calendar days
#   - exit_date = first available SPY trading date on or after exit_target_date

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CALL_SIGNAL_PANEL_PATH = (
    PROCESSED_DIR / "call_sleeve_30d_excel_vrp_signal_panel_v1.parquet"
)

SPY_EOD_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "market_data"
    / "spy_eod_prices_v1.parquet"
)

OUTPUT_PANEL_PATH = (
    PROCESSED_DIR / "call_sleeve_30d_excel_vrp_expiry_outcome_panel_v1.parquet"
)

print("=" * 100)
print("Cell 1F — Add 30D held-to-expiration outcome")
print("=" * 100)
print(f"Call signal panel:  {CALL_SIGNAL_PANEL_PATH}")
print(f"SPY EOD prices:     {SPY_EOD_PATH}")
print(f"Output:             {OUTPUT_PANEL_PATH}")
print()

# --------------------------------------------------------------------------------------
# Load inputs
# --------------------------------------------------------------------------------------

panel = pd.read_parquet(CALL_SIGNAL_PANEL_PATH)
spy = pd.read_parquet(SPY_EOD_PATH)

panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce")

# Flexible SPY date / close detection
spy_date_candidates = ["date", "trade_date", "asof_date", "timestamp"]
spy_close_candidates = ["spy_close", "close", "adj_close", "Close", "Adj Close"]

spy_date_col = next((c for c in spy_date_candidates if c in spy.columns), None)
spy_close_col = next((c for c in spy_close_candidates if c in spy.columns), None)

if spy_date_col is None:
    raise ValueError(f"Could not detect SPY date column. Columns: {list(spy.columns)}")

if spy_close_col is None:
    raise ValueError(f"Could not detect SPY close column. Columns: {list(spy.columns)}")

spy_prices = spy[[spy_date_col, spy_close_col]].copy()
spy_prices = spy_prices.rename(columns={spy_date_col: "spy_date", spy_close_col: "expiry_spy_close"})
spy_prices["spy_date"] = pd.to_datetime(spy_prices["spy_date"], errors="coerce")
spy_prices = spy_prices.dropna(subset=["spy_date"]).sort_values("spy_date").reset_index(drop=True)

# Guard against duplicate dates
dup_spy_dates = spy_prices["spy_date"].duplicated().sum()
if dup_spy_dates > 0:
    raise ValueError(f"SPY EOD price file has duplicate dates: {dup_spy_dates:,}")

# --------------------------------------------------------------------------------------
# Validate required panel columns
# --------------------------------------------------------------------------------------

required_panel_cols = [
    "trade_date",
    "spy_close",
    "call_signal_pass",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
    "call_spread_width_raw",
    "vix_style_vol_pct",
    "rv21d_vol_pct",
    "rsi14",
    "call_excel_vrp_log",
    "call_excel_vrp_z_3m",
    "call_excel_vrp_z_1y",
]

missing_panel_cols = [c for c in required_panel_cols if c not in panel.columns]
if missing_panel_cols:
    raise ValueError(f"Missing required panel columns: {missing_panel_cols}")

panel = panel.sort_values("trade_date").reset_index(drop=True)

# --------------------------------------------------------------------------------------
# Exit-date alignment
# --------------------------------------------------------------------------------------

panel["exit_target_date"] = panel["trade_date"] + pd.Timedelta(days=30)

# merge_asof finds the first spy_date >= exit_target_date
aligned = pd.merge_asof(
    panel.sort_values("exit_target_date"),
    spy_prices.sort_values("spy_date"),
    left_on="exit_target_date",
    right_on="spy_date",
    direction="forward",
)

aligned = aligned.rename(columns={"spy_date": "exit_date"})
aligned = aligned.sort_values("trade_date").reset_index(drop=True)

aligned["exit_calendar_days"] = (
    aligned["exit_date"] - aligned["trade_date"]
).dt.days

aligned["exit_trading_days_approx"] = np.nan

# Approximate trading day count between trade_date exclusive and exit_date inclusive
spy_calendar = spy_prices[["spy_date"]].copy()
spy_dates = spy_calendar["spy_date"].tolist()

date_to_idx = {d: i for i, d in enumerate(spy_dates)}

trade_idx = []
exit_idx = []

for _, row in aligned.iterrows():
    td = row["trade_date"]
    xd = row["exit_date"]
    trade_idx.append(date_to_idx.get(td, np.nan))
    exit_idx.append(date_to_idx.get(xd, np.nan))

aligned["trade_date_spy_index"] = trade_idx
aligned["exit_date_spy_index"] = exit_idx
aligned["exit_trading_days_approx"] = (
    aligned["exit_date_spy_index"] - aligned["trade_date_spy_index"]
)

# --------------------------------------------------------------------------------------
# Expiry intrinsic / win outcome
# --------------------------------------------------------------------------------------

aligned["short_call_intrinsic"] = np.maximum(
    aligned["expiry_spy_close"] - aligned["short_call_1sd_raw"],
    0.0,
)

aligned["long_call_intrinsic"] = np.maximum(
    aligned["expiry_spy_close"] - aligned["long_call_3sd_raw"],
    0.0,
)

# Short 1SD call, long 3SD call. Positive number here is terminal loss before premium.
aligned["terminal_spread_intrinsic_before_credit"] = (
    aligned["short_call_intrinsic"] - aligned["long_call_intrinsic"]
)

aligned["expired_short_call_otm"] = (
    aligned["expiry_spy_close"] <= aligned["short_call_1sd_raw"]
)

aligned["spread_intrinsic_zero"] = (
    aligned["terminal_spread_intrinsic_before_credit"] <= 0
)

# Basic outcome buckets before premium
aligned["expiry_outcome_bucket"] = np.select(
    [
        aligned["expiry_spy_close"] <= aligned["short_call_1sd_raw"],
        (aligned["expiry_spy_close"] > aligned["short_call_1sd_raw"])
        & (aligned["expiry_spy_close"] < aligned["long_call_3sd_raw"]),
        aligned["expiry_spy_close"] >= aligned["long_call_3sd_raw"],
    ],
    [
        "short_call_otm_full_credit_before_costs",
        "between_short_and_long_partial_intrinsic_loss",
        "above_long_call_max_intrinsic_loss",
    ],
    default="missing",
)

# --------------------------------------------------------------------------------------
# Selected baseline trades
# --------------------------------------------------------------------------------------

trade_eligible_cols = [
    "spy_close",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
    "call_spread_width_raw",
    "expiry_spy_close",
    "exit_date",
]

aligned["trade_outcome_eligible"] = (
    aligned["call_signal_pass"]
    & aligned[trade_eligible_cols].notna().all(axis=1)
)

selected = aligned.loc[aligned["trade_outcome_eligible"]].copy()

# --------------------------------------------------------------------------------------
# Summaries
# --------------------------------------------------------------------------------------

total_signal_count = int(aligned["call_signal_pass"].sum())
eligible_selected_count = int(len(selected))

expired_otm_count = int(selected["expired_short_call_otm"].sum())
intrinsic_zero_count = int(selected["spread_intrinsic_zero"].sum())

win_rate_short_otm = (
    expired_otm_count / eligible_selected_count
    if eligible_selected_count
    else np.nan
)

win_rate_zero_intrinsic = (
    intrinsic_zero_count / eligible_selected_count
    if eligible_selected_count
    else np.nan
)

outcome_bucket_summary = (
    selected["expiry_outcome_bucket"]
    .value_counts(dropna=False)
    .rename_axis("expiry_outcome_bucket")
    .reset_index(name="count")
)

outcome_bucket_summary["pct"] = (
    outcome_bucket_summary["count"] / eligible_selected_count
    if eligible_selected_count
    else np.nan
)

year_summary = (
    selected
    .assign(year=selected["trade_date"].dt.year)
    .groupby("year", as_index=False)
    .agg(
        selected_trades=("trade_date", "count"),
        expired_short_call_otm=("expired_short_call_otm", "sum"),
        spread_intrinsic_zero=("spread_intrinsic_zero", "sum"),
        avg_terminal_intrinsic_before_credit=("terminal_spread_intrinsic_before_credit", "mean"),
        max_terminal_intrinsic_before_credit=("terminal_spread_intrinsic_before_credit", "max"),
        avg_exit_calendar_days=("exit_calendar_days", "mean"),
        avg_exit_trading_days_approx=("exit_trading_days_approx", "mean"),
    )
)

if not year_summary.empty:
    year_summary["short_call_otm_win_rate"] = (
        year_summary["expired_short_call_otm"] / year_summary["selected_trades"]
    )
    year_summary["zero_intrinsic_win_rate"] = (
        year_summary["spread_intrinsic_zero"] / year_summary["selected_trades"]
    )

# Top intrinsic losses before premium
worst_intrinsic = selected.sort_values(
    "terminal_spread_intrinsic_before_credit",
    ascending=False,
).head(20)

# --------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

audit_panel_path = AUDIT_DIR / f"01F_call_sleeve_30d_expiry_outcome_panel_{timestamp}.csv"
selected_path = AUDIT_DIR / f"01F_call_sleeve_30d_selected_expiry_outcomes_{timestamp}.csv"
bucket_summary_path = AUDIT_DIR / f"01F_call_sleeve_30d_expiry_bucket_summary_{timestamp}.csv"
year_summary_path = AUDIT_DIR / f"01F_call_sleeve_30d_expiry_year_summary_{timestamp}.csv"
worst_path = AUDIT_DIR / f"01F_call_sleeve_30d_worst_intrinsic_outcomes_{timestamp}.csv"
manifest_path = AUDIT_DIR / f"01F_call_sleeve_30d_expiry_outcome_manifest_{timestamp}.json"

aligned.to_parquet(OUTPUT_PANEL_PATH, index=False)
aligned.to_csv(audit_panel_path, index=False)
selected.to_csv(selected_path, index=False)
outcome_bucket_summary.to_csv(bucket_summary_path, index=False)
year_summary.to_csv(year_summary_path, index=False)
worst_intrinsic.to_csv(worst_path, index=False)

manifest = {
    "cell": "1F",
    "description": "30D Excel-style VRP call sleeve held-to-expiration outcome validation",
    "created_at": timestamp,
    "call_signal_panel_path": str(CALL_SIGNAL_PANEL_PATH),
    "spy_eod_path": str(SPY_EOD_PATH),
    "output_panel_path": str(OUTPUT_PANEL_PATH),
    "audit_panel_path": str(audit_panel_path),
    "selected_path": str(selected_path),
    "bucket_summary_path": str(bucket_summary_path),
    "year_summary_path": str(year_summary_path),
    "worst_path": str(worst_path),
    "exit_convention": "first available SPY trading date on or after trade_date + 30 calendar days",
    "total_signal_count": total_signal_count,
    "eligible_selected_count": eligible_selected_count,
    "expired_otm_count": expired_otm_count,
    "intrinsic_zero_count": intrinsic_zero_count,
    "win_rate_short_call_otm": win_rate_short_otm,
    "win_rate_zero_intrinsic": win_rate_zero_intrinsic,
    "date_min": str(aligned["trade_date"].min().date()),
    "date_max": str(aligned["trade_date"].max().date()),
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# --------------------------------------------------------------------------------------
# Console output
# --------------------------------------------------------------------------------------

print("Built 30D expiry outcome panel")
print(f"Rows:                                {len(aligned):,}")
print(f"Date range:                          {aligned['trade_date'].min().date()} to {aligned['trade_date'].max().date()}")
print(f"SPY EOD date range:                   {spy_prices['spy_date'].min().date()} to {spy_prices['spy_date'].max().date()}")
print()
print("Exit convention:")
print("  exit_target_date = trade_date + 30 calendar days")
print("  exit_date = first available SPY trading date on or after exit_target_date")
print()
print("Selected baseline trades:")
print(f"  Total call signals:                 {total_signal_count:,}")
print(f"  Eligible selected trades:           {eligible_selected_count:,}")
print(f"  Missing selected outcomes:          {total_signal_count - eligible_selected_count:,}")
print()
print("Expiration outcome before premium:")
print(f"  Short call OTM count:               {expired_otm_count:,}")
print(f"  Short call OTM win rate:            {win_rate_short_otm:.2%}")
print(f"  Zero spread intrinsic count:        {intrinsic_zero_count:,}")
print(f"  Zero spread intrinsic win rate:     {win_rate_zero_intrinsic:.2%}")
print()
print("Exit timing sanity:")
print(f"  Avg calendar days:                  {selected['exit_calendar_days'].mean():.2f}" if len(selected) else "  Avg calendar days:                  n/a")
print(f"  Min calendar days:                  {selected['exit_calendar_days'].min():.0f}" if len(selected) else "  Min calendar days:                  n/a")
print(f"  Max calendar days:                  {selected['exit_calendar_days'].max():.0f}" if len(selected) else "  Max calendar days:                  n/a")
print(f"  Avg trading days approx:            {selected['exit_trading_days_approx'].mean():.2f}" if len(selected) else "  Avg trading days approx:            n/a")
print()
print("Saved:")
print(f"  Output parquet:                     {OUTPUT_PANEL_PATH}")
print(f"  Audit panel CSV:                    {audit_panel_path}")
print(f"  Selected outcomes CSV:              {selected_path}")
print(f"  Bucket summary CSV:                 {bucket_summary_path}")
print(f"  Year summary CSV:                   {year_summary_path}")
print(f"  Worst intrinsic CSV:                {worst_path}")
print(f"  Manifest JSON:                      {manifest_path}")

print()
print("=" * 100)
print("Outcome bucket summary")
print("=" * 100)
display(outcome_bucket_summary)

print()
print("=" * 100)
print("Year summary")
print("=" * 100)
display(year_summary)

print()
print("=" * 100)
print("Worst intrinsic outcomes before premium")
print("=" * 100)
display(
    worst_intrinsic[
        [
            "trade_date",
            "exit_date",
            "spy_close",
            "expiry_spy_close",
            "vix_style_vol_pct",
            "short_call_1sd_raw",
            "long_call_3sd_raw",
            "terminal_spread_intrinsic_before_credit",
            "expired_short_call_otm",
            "expiry_outcome_bucket",
            "call_excel_vrp_z_3m",
            "call_excel_vrp_z_1y",
            "rsi14",
        ]
    ]
)

print()
print("=" * 100)
print("Latest selected outcomes")
print("=" * 100)
display(
    selected.tail(20)[
        [
            "trade_date",
            "exit_date",
            "spy_close",
            "expiry_spy_close",
            "short_call_1sd_raw",
            "long_call_3sd_raw",
            "expired_short_call_otm",
            "terminal_spread_intrinsic_before_credit",
            "expiry_outcome_bucket",
            "call_excel_vrp_z_3m",
            "call_excel_vrp_z_1y",
            "rsi14",
        ]
    ]
)

Cell 1F — Add 30D held-to-expiration outcome
Call signal panel:  C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_30d_excel_vrp_signal_panel_v1.parquet
SPY EOD prices:     C:\Users\patri\vrp_project\data\processed\market_data\spy_eod_prices_v1.parquet
Output:             C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_30d_excel_vrp_expiry_outcome_panel_v1.parquet

Built 30D expiry outcome panel
Rows:                                1,635
Date range:                          2020-01-02 to 2026-07-08
SPY EOD date range:                   2018-01-02 to 2026-07-08

Exit convention:
  exit_target_date = trade_date + 30 calendar days
  exit_date = first available SPY trading date on or after exit_target_date

Selected baseline trades:
  Total call signals:                 285
  Eligible selected trades:           285
  Missing selected outcomes:          0

Expiration outcome before premium:
  Short call OTM count:               274
  Short

,expiry_outcome_bucket,count,pct
0,short_call_otm_full_credit_before_costs,274,0.961404
1,between_short_and_long_partial_intrinsic_loss,11,0.038596



Year summary


,year,selected_trades,expired_short_call_otm,spread_intrinsic_zero,avg_terminal_intrinsic_before_credit,max_terminal_intrinsic_before_credit,avg_exit_calendar_days,avg_exit_trading_days_approx,short_call_otm_win_rate,zero_intrinsic_win_rate
0,2020,1,1,1,0.000000,0.000000,32.000000,20.000000,1.000000,1.000000
1,2021,72,72,72,0.000000,0.000000,30.638889,21.069444,1.000000,1.000000
2,2023,58,58,58,0.000000,0.000000,30.551724,20.879310,1.000000,1.000000
3,2024,69,59,59,0.505196,8.113880,30.565217,20.942029,0.855072,0.855072
4,2025,62,61,61,0.011641,0.721755,30.580645,21.387097,0.983871,0.983871
5,2026,23,23,23,0.000000,0.000000,30.695652,20.739130,1.000000,1.000000



Worst intrinsic outcomes before premium


,trade_date,exit_date,spy_close,expiry_spy_close,vix_style_vol_pct,short_call_1sd_raw,long_call_3sd_raw,terminal_spread_intrinsic_before_credit,expired_short_call_otm,expiry_outcome_bucket,call_excel_vrp_z_3m,call_excel_vrp_z_1y,rsi14
1015,2024-01-16,2024-02-15,474.93,502.01,13.833737,493.896120,531.828359,8.113880,False,between_short_and_long_partial_intrinsic_loss,0.579010,0.460866,61.060682
1112,2024-06-04,2024-07-05,528.39,554.64,13.113000,548.391660,588.394979,6.248340,False,between_short_and_long_partial_intrinsic_loss,1.352290,1.135226,58.749190
1116,2024-06-10,2024-07-10,535.66,561.32,12.685356,555.275585,594.506755,6.044415,False,between_short_and_long_partial_intrinsic_loss,0.931077,0.806730,65.723151
1020,2024-01-23,2024-02-22,484.86,507.50,12.590940,502.483164,537.729491,5.016836,False,between_short_and_long_partial_intrinsic_loss,0.435432,0.205755,70.146981
1021,2024-01-24,2024-02-23,485.39,507.85,13.044339,503.667731,540.223192,4.182269,False,between_short_and_long_partial_intrinsic_loss,0.890623,0.713893,70.526950
1115,2024-06-07,2024-07-08,534.01,555.28,12.217980,552.844676,590.514029,2.435324,False,between_short_and_long_partial_intrinsic_loss,0.641670,0.511312,64.198489
1114,2024-06-06,2024-07-08,534.66,555.28,12.543040,554.019311,592.737933,1.260689,False,between_short_and_long_partial_intrinsic_loss,0.850812,0.697687,65.366477
1376,2025-06-25,2025-07-25,607.12,637.10,16.694152,636.378245,694.894735,0.721755,False,between_short_and_long_partial_intrinsic_loss,1.053747,0.366261,65.107321
1017,2024-01-18,2024-02-20,476.49,496.76,14.212374,496.039236,535.137707,0.720764,False,between_short_and_long_partial_intrinsic_loss,0.530435,0.386353,61.396740
1113,2024-06-05,2024-07-05,534.67,554.64,12.607818,554.129655,593.048964,0.510345,False,between_short_and_long_partial_intrinsic_loss,0.917039,0.745720,65.564999



Latest selected outcomes


,trade_date,exit_date,spy_close,expiry_spy_close,short_call_1sd_raw,long_call_3sd_raw,expired_short_call_otm,terminal_spread_intrinsic_before_credit,expiry_outcome_bucket,call_excel_vrp_z_3m,call_excel_vrp_z_1y,rsi14
1515,2026-01-13,2026-02-12,693.77,681.27,725.760970,789.742909,True,0.0,short_call_otm_full_credit_before_costs,1.495661,0.818251,60.666193
1524,2026-01-27,2026-02-26,695.49,689.30,728.016084,793.068253,True,0.0,short_call_otm_full_credit_before_costs,0.413102,0.363007,58.561546
1525,2026-01-28,2026-02-27,695.42,685.99,728.648411,795.105232,True,0.0,short_call_otm_full_credit_before_costs,0.514336,0.417457,58.489398
1594,2026-05-07,2026-06-08,731.58,739.22,767.629464,839.728391,True,0.0,short_call_otm_full_credit_before_costs,0.480828,0.386089,72.092432
1595,2026-05-08,2026-06-08,737.62,739.22,774.150383,847.211148,True,0.0,short_call_otm_full_credit_before_costs,0.476725,0.363873,74.578135
1596,2026-05-11,2026-06-10,739.30,725.43,778.486584,856.859751,True,0.0,short_call_otm_full_credit_before_costs,0.765354,0.656345,75.115221
1597,2026-05-12,2026-06-11,738.18,737.76,776.366124,852.738372,True,0.0,short_call_otm_full_credit_before_costs,0.670494,0.582669,73.683428
1598,2026-05-13,2026-06-12,742.31,741.75,780.618604,857.235813,True,0.0,short_call_otm_full_credit_before_costs,0.820151,0.750706,75.514938
1599,2026-05-14,2026-06-15,748.17,754.83,785.571786,860.375358,True,0.0,short_call_otm_full_credit_before_costs,0.677639,0.619319,77.713863
1600,2026-05-15,2026-06-15,739.17,754.83,778.091086,855.933257,True,0.0,short_call_otm_full_credit_before_costs,0.364217,0.293261,67.146096
